# CodeSnap - Natural Language to Code
Package: com.appwhiz.codesnap
Run all cells to start the app

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import textwrap, json, os
print('✅ Ready - Widgets loaded')


In [ ]:
LANGUAGES = {
    'Python': {'file': 'solution.py', 'comment': '#', 'color': '#3776AB'},
    'C++': {'file': 'solution.cpp', 'comment': '//', 'color': '#00599C'},
    'C': {'file': 'solution.c', 'comment': '//', 'color': '#A97BFF'},
    'Java': {'file': 'Solution.java', 'comment': '//', 'color': '#ED8B00'},
    'JavaScript': {'file': 'solution.js', 'comment': '//', 'color': '#F7DF1E'},
    'C#': {'file': 'Solution.cs', 'comment': '//', 'color': '#68217A'},
    'Go': {'file': 'main.go', 'comment': '//', 'color': '#00ADD8'},
    'Kotlin': {'file': 'Solution.kt', 'comment': '//', 'color': '#7F52FF'},
    'Rust': {'file': 'main.rs', 'comment': '//', 'color': '#DEA584'}
}

def translate(text, lang):
    t=text.lower(); c=LANGUAGES[lang]['comment']; head=f"{c} From: {text[:100]}\n{c} Lang: {lang}\n\n"
    def w(b): return head + textwrap.dedent(b).strip() + "\n"
    if 'fizzbuzz' in t:
        if lang=='Python': return w('def fizzbuzz(n=100):\n    for i in range(1,n+1):\n        if i%15==0: print("FizzBuzz")\n        elif i%3==0: print("Fizz")\n        elif i%5==0: print("Buzz")\n        else: print(i)\nfizzbuzz()')
        return w(f'{c} FizzBuzz in {lang}')
    if 'prime' in t:
        if lang=='Python': return w('def is_prime(n):\n    if n<=1: return False\n    if n<=3: return True\n    if n%2==0 or n%3==0: return False\n    i=5\n    while i*i<=n:\n        if n%i==0 or n%(i+2)==0: return False\n        i+=6\n    return True\nprint(is_prime(17))')
        return w(f'{c} Prime check in {lang}')
    if 'sort' in t and lang=='Python': return w('nums=[64,34,25,12,22,11,90]\nprint(f"Sorted: {sorted(nums)}")')
    if ('largest' in t or 'max' in t) and lang=='Python': return w('print(max([3,7,2,9,4]))  # 9')
    if 'factorial' in t and lang=='Python': return w('def fact(n): return 1 if n<=1 else n*fact(n-1)\nprint(fact(5))  # 120')
    if 'fibonacci' in t and lang=='Python': return w('def fib(n):\n    a,b=0,1; r=[]\n    for _ in range(n): r.append(a); a,b=b,a+b\n    return r\nprint(fib(10))')
    if 'calculator' in t and lang=='Python': return w('class Calculator:\n    def add(self,a,b): return a+b\n    def subtract(self,a,b): return a-b\n    def multiply(self,a,b): return a*b\n    def divide(self,a,b): return a/b\ncalc=Calculator()\nprint(calc.add(10,5))')
    if lang=='Python': return w(f'# {text}\ndef solve(data=[1,2,3,4,5]):\n    return [x*2 for x in data]\nprint(solve())')
    return w(f'{c} {text} -> {lang} implementation')
print('✅ Translator ready - 9 languages')


In [ ]:
HISTORY_FILE='codesnap_history.json'
history=[]
if os.path.exists(HISTORY_FILE):
    try: history=json.load(open(HISTORY_FILE))
    except: history=[]
def save_history(): json.dump(history[:20], open(HISTORY_FILE,'w'))

input_box=widgets.Textarea(placeholder='e.g. check if prime, fizzbuzz, sort a list...', description='Scratch Pad:', layout=widgets.Layout(width='100%', height='110px'))
translate_btn=widgets.Button(description='✨ Translate', button_style='success', layout=widgets.Layout(width='180px', height='38px'))
clear_btn=widgets.Button(description='Clear', button_style='warning', layout=widgets.Layout(width='80px'))
lang_tabs=widgets.Tab()
code_outputs={}
for lang in LANGUAGES:
    code_outputs[lang]=widgets.Textarea(value='Your code will appear here...', layout=widgets.Layout(width='100%', height='380px'))
lang_tabs.children=tuple(code_outputs.values())
for i,lang in enumerate(LANGUAGES): lang_tabs.set_title(i, lang)
history_box=widgets.VBox()
out=widgets.Output()

def refresh_history():
    items=[]
    if history:
        items.append(widgets.HTML('<b>Recent Prompts (click X to delete):</b>'))
        for idx,h in enumerate(history):
            b1=widgets.Button(description=h[:42], layout=widgets.Layout(width='65%'))
            b2=widgets.Button(description='❌ Delete', button_style='danger', layout=widgets.Layout(width='25%'))
            def make_load(txt):
                return lambda x: (setattr(input_box,'value',txt), on_translate(None))
            def make_del(i):
                return lambda x: (history.pop(i), save_history(), refresh_history())
            b1.on_click(make_load(h))
            b2.on_click(make_del(idx))
            items.append(widgets.HBox([b1,b2]))
        cb=widgets.Button(description='Clear All History', button_style='danger', layout=widgets.Layout(width='200px'))
        cb.on_click(lambda x: (history.clear(), save_history(), refresh_history()))
        items.append(cb)
    history_box.children=tuple(items)

def on_translate(b):
    text=input_box.value.strip()
    if not text: return
    if text not in history:
        history.insert(0,text); save_history()
    refresh_history()
    for lang,area in code_outputs.items(): area.value=translate(text, lang)
    with out:
        clear_output(); print(f'✅ Translated "{text[:30]}..." to {len(LANGUAGES)} languages')

translate_btn.on_click(on_translate)
clear_btn.on_click(lambda b: setattr(input_box,'value',''))
refresh_history()
display(widgets.VBox([widgets.HTML('<h2>🚀 CodeSnap - Type in Scratch Pad, pick language tab on top</h2>'), input_box, widgets.HBox([translate_btn, clear_btn]), history_box, lang_tabs, out]))


In [ ]:
def save_and_run(b):
    idx=lang_tabs.selected_index
    lang=list(LANGUAGES.keys())[idx]
    code=code_outputs[lang].value
    fname=LANGUAGES[lang]['file']
    open(fname,'w').write(code)
    print(f'💾 Saved as {fname}')
    if lang=='Python':
        print('\n--- Running Python code ---')
        try: exec(code)
        except Exception as e: print(f'Error: {e}')
btn=widgets.Button(description='💾 Save & Run Current Tab', button_style='primary', layout=widgets.Layout(width='250px', height='40px'))
btn.on_click(save_and_run)
display(btn)
